# NumPy 复习与易错点（个人定制版）

> 用法：从头到尾过一遍，**红色易错点**重点看；每段代码先自己想输出，再运行核对。
> 学习区间：2026-09-23 ~ 09-28，环境 conda `ai_learn`（Python 3.10 + NumPy 2.2）。

In [ ]:
import numpy as np
print(np.__version__)

2.2.5


## 1. ndarray 到底是什么

一个 ndarray = **一块连续的数据区** + **一个数组头（说明书）**。

数组头里记录：
- `data`：数据区起始地址
- `shape`：各维长度（数字个数=维度数）
- `strides`：各维"走一步"要跨过多少**字节**
- `dtype`：元素类型（决定每个元素占几字节）

**和 list 的根本区别**

| | list | ndarray |
|---|---|---|
| 内存 | 分散，格子里是指针，指向独立 Python 对象 | 连续的原始字节 |
| 类型 | 可混装 | 必须同类型 |
| `+` | 拼接 | 逐元素加 |
| `*2` | 重复 | 逐元素乘 |
| 统计/广播 | 没有 | 有 |

> 一维 ndarray 只是"长得像 list"，本质是**数学向量（计算对象）**，不是收纳盒。

In [2]:
lst=[1,2,3]; arr=np.array([1,2,3])
print("list +:", lst+[4,5,6])      # 拼接
print("array+:", arr+np.array([4,5,6]))  # 逐元素
print("array*2:", arr*2)

list +: [1, 2, 3, 4, 5, 6]
array+: [5 7 9]
array*2: [2 4 6]


## 2. 创建数组函数表

| 函数 | 作用 |
|---|---|
| `np.array(x)` | list/tuple/变量 -> 数组 |
| `np.arange(a,b,s)` | 等差序列（不含b） |
| `np.linspace(a,b,n)` | 均匀 n 个点（含b） |
| `np.zeros(shape)` | 全 0 |
| `np.ones(shape)` | 全 1 |
| `np.full(shape,v)` | 填充指定值 |
| `np.eye(n)` | n 阶单位矩阵 |
| `np.diag(x)` | 取对角线 / 构造对角阵 |
| `np.random.rand(*shape)` | [0,1) 随机 |
| `np.random.randint(lo,hi,shape)` | 随机整数（不含 hi） |
| `np.random.seed(n)` | 固定随机种子 |

> **易错①** `np.zeros/ones` 默认是 **float64**，要整数必须 `dtype=int`。
> **易错②** 题目指定 seed 时，**先 `np.random.seed(n)` 再造随机数**，否则结果对不上、不可复现。

In [3]:
print(np.zeros((2,3)).dtype)            # float64
print(np.zeros((2,3), dtype=int).dtype) # int64
np.random.seed(0)
print(np.random.randint(0,101,(5,5))[0])# 固定首行 [44 47 64 67 67]

float64
int64
[44 47 64 67 67]


## 3. 核心属性

| 属性 | 含义 |
|---|---|
| `shape` | 各维长度（元组） |
| `ndim` | 维度数 |
| `size` | **元素总个数**（不是字节数！） |
| `dtype` | 元素类型 |
| `nbytes` | 占用总字节数 = size × 每项字节 |
| `strides` | 各维步长（字节） |
| `T` | 转置 |

> **易错③** `size` 是元素个数（3×4=12），想看占多少内存用 `nbytes`。
> **易错④** `(3,)` 是一维；`(3,1)` 是二维；`(1,3)` 也是二维。括号层数=维度数。

In [4]:
a=np.arange(12).reshape(3,4)
print(a.shape, a.ndim, a.size, a.dtype, a.nbytes)
print(np.array([1,2,3]).shape, np.array([1,2,3])[:,None].shape)

(3, 4) 2 12 int64 96
(3,) (3, 1)


## 4. 三种索引（重点）

### 4.1 基础索引：整数 + 切片
- 逗号分隔维度：`a[行, 列]`，索引从 **0** 开始
- 切片 `start:stop:step`（不含 stop）
- 索引数量少于维度时，**末尾自动补 `:`**：`a[0]` = `a[0,:]`

### 4.2 布尔索引
- `a[条件]`：先算布尔掩码，再收集 True 的元素，**结果恒为一维**
- 多条件用 `& | ~`（不是 and/or），**每个条件加括号**：`(a>60)&(a<90)`

### 4.3 花式索引
- `a[[0,2,1]]`：方括号里是**整数列表**，按清单取行（可乱序、可重复）
- 注意区分：`a[0]` 取一行降维；`a[[0]]` 取一行但保持二维

In [5]:
a=np.arange(12).reshape(3,4)
print(a[1:3, 1:3])          # 切片
print(a[(a>3)&(a<9)])       # 布尔，一维
print(a[[2,0]])             # 花式，按清单取行
print(a[0].shape, a[[0]].shape)  # (4,) vs (1,4)

[[ 5  6]
 [ 9 10]]
[4 5 6 7 8]
[[ 8  9 10 11]
 [ 0  1  2  3]]
(4,) (1, 4)


## 5. 视图与副本（最高频易错点）

**先分清三个动作**
```
b = a          # 贴标签：连数组头都是同一个，a is b -> True
b = a.view()   # 新数组头，但共享数据区
b = a.copy()   # 数组头 + 数据区全部新建，完全独立
```

### 视图/副本判定口诀（背下来）

| 方括号/操作里是 | 结果 |
|---|---|
| 冒号切片 `a[:,1:]`、`a[1:4]` | **视图** |
| 条件/布尔 `a[a>5]` | **副本** |
| 整数列表 `a[[0,2]]` | **副本** |
| `reshape / ravel / T / newaxis`（可行时） | **视图** |
| `flatten / sort(返回值) / 布尔&花式` | **副本** |

> **易错⑤（你犯过2次）布尔索引恒为副本**：要先 nonzero 定位、把分散元素收集进新内存。
> **易错⑥ newaxis 是视图**：只加括号改 shape，数据指针不动。
>
> 判定方法：`np.shares_memory(a,b)`（True=共享=视图）；或改个值看原数组变不变。

In [6]:
a=np.arange(12).reshape(3,4)
print("布尔 shares:", np.shares_memory(a, a[a>5]))   # False
print("newaxis shares:", np.shares_memory(a, a[:,None])) # True
print("切片 shares:", np.shares_memory(a, a[:,1:]))  # True
b=a.copy(); b[0,0]=99
print("copy后原数组不变:", a[0,0])

布尔 shares: False
newaxis shares: True
切片 shares: True
copy后原数组不变: 0


## 6. 形状操作

| 操作 | 作用 | 视图/副本 |
|---|---|---|
| `reshape(m,n)` | 改形状（元素数不变） | 视图（可行时） |
| `ravel()` | 展平成一维 | 视图 |
| `flatten()` | 展平成一维 | 副本 |
| `T` | 转置（步长交换，数据不动） | 视图 |
| `np.newaxis / None` | 插入长度1的维 | 视图 |
| `squeeze()` | 去掉长度1的维 | 视图 |
| `concatenate/hstack/vstack` | 拼接（要真加元素） | 新数组 |

> **易错⑦ reshape 不改原数组**，它返回新（视图）对象，要接住：`b=a.reshape(...)`。
> **易错⑧ newaxis 增加的是"维度数"不是"加长现有维度"**：
> `(3,2)` 加 newaxis -> `(3,2,1)`（三维），不会变成 `(3,3)`。
> 口诀：newaxis 加盖楼层（维度+1，元素不增）；hstack 才是多隔房间。

In [7]:
a=np.arange(6)
b=a.reshape(2,3)        # 要接住
print(a.shape, b.shape)
print(np.arange(6).reshape(2,3)[:,None].shape)  # (2,3,1)
x=np.array([1,2,3])
print(x[None,:].shape, x[:,None].shape, x[None,:].squeeze().shape)

(6,) (2, 3)
(2, 1, 3)
(1, 3) (3, 1) (3,)


## 7. 广播机制（重点易错）

**三条规则（按顺序）**
1. **右对齐**：从最右边的维度开始比较
2. **缺维补 1**：维度少的，在**最左边**补 1（从不插中间！）
3. **遇 1 拉伸**：某维长度为 1，就复制拉伸成对方长度；两数都≠1 且不相等 -> 报错

**常见组合**

| 形状 | 能否 | 结果 |
|---|---|---|
| `(3,4)+(4,)` | ✓ 补(1,4) | (3,4) |
| `(3,4)+(3,)` | ✗ 右维4≠3 | 报错 |
| `(3,4)+(3,1)` | ✓ | (3,4) |
| `(4,1)+(3,)` | ✓ 补(1,3) | (4,3) |
| `(2,3,4)+(2,4)` | ✗ 补(1,2,4)，中间3≠2 | **报错** |

> **易错⑨（你错过）`(2,3,4)+(2,4)` 自动广播报错**：补维只补最左边成 `(1,2,4)`，
> 中间维 3 vs 2 冲突。想插中间必须**手动** `b[:,None,:]` 成 `(2,1,4)`，结果 `(2,3,4)`。
>
> 价值：免写循环，"列向量+行向量"一行算出整张表（如距离矩阵、外积、减均值）。

In [8]:
print((np.ones((3,4))+np.array([10,20,30,40])).shape)  # (3,4)
col=np.array([[10],[20],[30]]); row=np.array([1,2,3])
print(col+row)   # (3,1)+(1,3)->(3,3)
try:
    np.ones((2,3,4))+np.ones((2,4))
except ValueError as e:
    print("报错:", str(e)[:40])
print("手动插中间:", (np.ones((2,3,4))+np.ones((2,4))[:,None,:]).shape)

(3, 4)
[[11 12 13]
 [21 22 23]
 [31 32 33]]
报错: operands could not be broadcast together
手动插中间: (2, 3, 4)


## 8. 运算、统计、排序

- 逐元素：`+ - * / **`（`*` 逐元素，矩阵乘用 `@`）
- 通用函数：`sqrt exp log abs round sum mean std var min max`
- 统计方向：`axis=n` 消灭第 n 维；`keepdims=True` 把该维保留成 1（省一步 newaxis）
- 定位：`argmax/argmin`（最值的**索引**）；`argsort`（排序后的**索引清单**）
- 条件：`np.where(cond,x,y)`（选择，返回新数组）；单参 `np.where(mask)[0]`（取 True 行号）
- 截断：`np.clip(a,lo,hi)`（处理异常/脏数据）

> **易错⑩ "有任意一门<60"用 `(a<60).any(axis=1)`，不是判断平均分<60。**
> any=有没有True；all=是不是全True。

In [9]:
a=np.array([[90,50,80],[70,60,75]])
print(a.mean(axis=0))                 # 列均值
print(a.sum(axis=1, keepdims=True).shape)  # (2,1)
print((a>=60).all(axis=1))            # [False True]
print((a<60).any(axis=1))             # [True False]
print(np.argsort(a[:,0]))             # 按第0列排序的行号

[80.  55.  77.5]
(2, 1)
[False  True]
[ True False]
[1 0]


## 9. 布尔即 0/1（高频技巧）

布尔值参与算术时 **True=1，False=0**：

```python
mask = a >= 60
mask.sum()    # True 的个数 = 及格人数
mask.mean()   # True 的比例 = 及格率
```

`(a>=60).mean(axis=0)` 能算及格率，就是因为 1 的个数/总数 = 比例。

In [10]:
np.random.seed(42); a=np.random.randint(50,101,(10,5))
print("及格人数(列):", (a>=60).sum(axis=0))
print("及格率(列):", (a>=60).mean(axis=0).round(1))

及格人数(列): [9 8 8 8 9]
及格率(列): [0.9 0.8 0.8 0.8 0.9]


## 10. 线性代数（固定套路，不依赖高数）

```python
A @ B            # 矩阵乘法：(M,K)@(K,N)->(M,N)
np.linalg.solve(A,b)  # 解 Ax=b（首选，比 inv 稳）
np.linalg.inv(A)      # 逆矩阵（方阵且 det≠0）
np.linalg.det(A)      # 行列式
```
验证：`A @ inv(A) ≈ I`（单位矩阵）。

In [11]:
A=np.array([[2.,1],[1,3]]); b=np.array([4.,5])
x=np.linalg.solve(A,b)
print("x=",x, " 验证 A@x=",A@x)
print((A@np.linalg.inv(A)).round(6))

x= [1.4 1.2]  验证 A@x= [4. 5.]
[[ 1.  0.]
 [-0.  1.]]


## 11. 个人易错红榜（考前只看这个）

1. **布尔索引 = 副本**（不是视图）——犯过 2 次
2. **newaxis = 视图**（不是副本）
3. **自动补维只补最左边，不插中间**；`(2,3,4)+(2,4)` 报错
4. `np.zeros/ones` 默认 float，整数加 `dtype=int`
5. 随机数先 `np.random.seed(n)`
6. `(3,)` 一维 ≠ `(3,1)` 二维；`size` 是元素数，字节数用 `nbytes`
7. "有任意一门<60" -> `any`，别拿平均分判断
8. `reshape` 不改原数组，要接住返回值
9. `argsort/argmax` 给的是**索引**，不是值
10. 链式 `表达式[...]`：先算表达式再索引；看不懂就拆变量

**一句总纲**：连续内存 + 数组头(shape/strides) 是一切的基础；
切片靠改 strides 出视图，条件/乱序靠复制出副本；广播靠右对齐+补1+拉伸免写循环。